# Hybrid LSTM + Random Forest — corrected experiment

Run cells in order from a fresh kernel. Open this notebook from the repository checkout; its first code cell locates the repository root.

**Protocol:** rolling one-hour-ahead forecasts, 168-hour history by default; 70% base training, 7.5% early stopping, 7.5% hybrid calibration and 15% test. Every model uses the same test timestamps. A plot of 168 successive one-hour forecasts is not a week-ahead forecast.

Supply an authorized CSV at `data/electricity_demand.csv`; it is excluded from this repository. No historical scores are reused. See `docs/CORRECTIONS.md` for changes and `forecasting/models.py` for the model implementation.


## 1. Environment
Install once in a clean Python 3.12 environment: `python -m pip install -r requirements.txt`. In Colab, upload and extract the entire project ZIP, change directory to its project folder, then run `%pip install -r requirements.txt` and restart the runtime if prompted. Do not upload only this notebook; it uses the shared code folder.

In [ ]:
from pathlib import Path
import os, sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'forecasting').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'forecasting').is_dir():
    raise RuntimeError('Run from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pathlib import Path
import os
from IPython.display import display, Image
from forecasting.data import load_data, prepare_data, split_boundaries, synthetic_data
from forecasting.experiment import run_experiment

MODEL = 'lstm_rf'
DATA_PATH = Path(os.environ.get("FORECAST_DATA", "data/electricity_demand.csv"))
OUTPUT_DIR = Path(os.environ.get("FORECAST_OUTPUT", "results"))
DEMO = os.environ.get("FORECAST_DEMO", "0") == "1"  # explicitly synthetic data only
SMOKE = os.environ.get("FORECAST_SMOKE", "0") == "1"  # software checks, not research scores
MAX_ROWS = int(os.environ["FORECAST_MAX_ROWS"]) if os.environ.get("FORECAST_MAX_ROWS") else None
EPOCHS = int(os.environ.get("FORECAST_EPOCHS", "1" if SMOKE else "50"))
ENCODER_LENGTH = 24 if SMOKE else 168
SEED = 42


## 2. Validate and inspect the data
Calendar fields are reconstructed from timestamps. Missing hour labels are recovered only for complete, ordered 24-row days with matching known labels. Historical missing demand/weather is filled forward, never backward. Missing demand labels are excluded from training targets and scoring.

In [ ]:
raw = synthetic_data(seed=SEED) if DEMO else load_data(DATA_PATH)
if MAX_ROWS is not None:
    raw = raw.iloc[:MAX_ROWS].copy()
data = prepare_data(raw)
cuts = split_boundaries(data, ENCODER_LENGTH)
print("SYNTHETIC DEMO" if DEMO else "REAL CSV", "| raw rows:", len(raw), "| prepared rows:", len(data))
print("Split endpoints:", cuts)
print("Test:", data.iloc[cuts[2]].datetime, "to", data.iloc[-1].datetime)
display(data.head())


## 3. Train and evaluate
Early stopping selects the best neural checkpoint. Hybrid residual models train on a separate calibration period. Predictions are joined to their target timestamps. Realized future weather is never assumed available. The test set is not used for fitting or model selection.

In [ ]:
result = run_experiment(MODEL, DATA_PATH, OUTPUT_DIR, demo=DEMO, smoke=SMOKE,
                        epochs=EPOCHS, encoder_length=ENCODER_LENGTH, seed=SEED, max_rows=MAX_ROWS)


## 4. Fresh results
Metrics cover only measured test demand. MAPE excludes zero-demand targets and reports the excluded count. Hybrid rows include the exact base model for comparison, plus a weekly seasonal-naive reference. Smoke-test scores are not final research results.

In [ ]:
display(result["metrics"])
display(Image(filename=str(result["run_dir"] / "forecast.png")))
print("Outputs saved to:", result["run_dir"].resolve())
print("Configuration and dependency versions:", result["manifest"])
